# Лабораторная работа № 3.1 — Интерполяция Лагранжа и Ньютона

Вариант 6. 

По четырём узлам строим многочлен для y=eˣ. Две формы записи одного многочлена должны дать одинаковое значение при x*=-0.5.

## Исходные данные

Числа ниже соответствуют файлу input.txt этой работы.

In [1]:
import math
import numpy as np
import matplotlib.pyplot as plt
from typing import List

rows = [[-2.0, -1.0, 0.0, 1.0], [-2.0, -1.0, 0.2, 1.0], [-0.5]]
sets, x_star = rows[:2], rows[2][0]
print('Наборы узлов:', sets, 'точка:', x_star)

Наборы узлов: [[-2.0, -1.0, 0.0, 1.0], [-2.0, -1.0, 0.2, 1.0]] точка: -0.5


## Две формы интерполяционного многочлена

Лагранж использует базисные произведения, Ньютон — разделённые разности. Для каждого набора узлов результаты совпадают.

In [2]:
def f(x):
    return math.exp(x)


def prod(points, x, i):
    res = 1
    for j in range(len(points)):
        if j != i:
            res *= (x - points[j])
    return res


def prod_to_print(points, i):
    prod = ""
    for j in range(len(points)):
        if i != j:
            prod += f"(x - {points[j]})"
    return prod


def Lagrange_interpolation(points, x):
    res = 0
    res_str = "L(x) = "
    for i in range(len(points)):
        f_prod = f(points[i]) / prod(points, points[i], i)
        res += f_prod * prod(points, x, i)

        sign = " + " if f_prod > 0 else ""
        res_str += f"{sign} {f_prod:.12g}*" + prod_to_print(points, i)

    return res, res_str


def Newton_interpolation(points, x):
    y = [f(p) for p in points]

    coefs = [y[i] for i in range(len(points))]
    for j in range(1, len(points)):
        for i in range(len(points) - 1, j - 1, -1):
            coefs[i] = float(coefs[i] - coefs[i - 1]) / float(points[i] - points[i - j])

    
    res = coefs[0]
    res_str = f"P(x) = {coefs[0]:.12g}"
    current_terms = []
    
    for i in range(1, len(coefs)):
        current_terms.append(f"(x - {points[i-1]:.12g})")
        term_str = "*".join(current_terms)
        p = 1
        for j in range(i):
            p *= x - points[j]
        res += coefs[i] * p
        
        sign = " + " if coefs[i] >= 0 else " - "
        res_str += f"{sign}{abs(coefs[i]):.12g}*{term_str}"
    
    return res, res_str

In [3]:
results = []
for xs in sets:
    lagrange, _ = Lagrange_interpolation(xs, x_star)
    newton, _ = Newton_interpolation(xs, x_star)
    error = abs(f(x_star)-lagrange)
    results.append((lagrange, newton, error))
    print('Узлы:', xs, 'L =', lagrange, 'N =', newton, 'ошибка =', error)

Узлы: [-2.0, -1.0, 0.0, 1.0] L = 0.5910811161779577 N = 0.5910811161779577 ошибка = 0.015449543534675758
Узлы: [-2.0, -1.0, 0.2, 1.0] L = 0.5839486640322064 N = 0.5839486640322064 ошибка = 0.02258199568042707


## Самопроверка

Почему при одинаковых узлах формы Лагранжа и Ньютона совпадают? Проверь численно.

In [4]:
assert all(abs(l-n) < 1e-12 for l,n,_ in results)
print('Точное eˣ* =', f(x_star))
print('Лучший набор:', 1 + min(range(2), key=lambda i: results[i][2]))

Точное eˣ* = 0.6065306597126334
Лучший набор: 1
